In [ ]:
import torch
import os
import numpy as np
import torch.nn as nn
from matplotlib import pyplot as plt
import matplotlib.colors as colors
import seaborn as sns
import json
import pandas as pd


colors = {"meta-llama/Llama-2-7b-hf":"#0081FB",
          "mistralai/Mistral-7B-v0.1":"#FFA500"}

names = {"meta-llama/Llama-2-7b-hf":"Llama2-7B",
        "mistralai/Mistral-7B-v0.1":"Mistral-7B"}

LMs = ["meta-llama/Llama-2-7b-hf", "mistralai/Mistral-7B-v0.1"] 
n_classes = {"rte":2,
             "arc_challenge":4,
             "openbookqa":4,
             "boolq":2,
             "arc_easy":4
             }

d_names = {
    "rte":"RTE",
    "arc_challenge":"ARC Challenge",
    "openbookqa":"OpenBookQA",
    "boolq":"BoolQ",
    "arc_easy":"ARC Easy"
}


all_res_model = {}
for model_name in LMs: 
    
    all_res = []
    
    base_results_path = "./acts/activations/"+ model_name+"/"    
    order_to_remove = np.load(base_results_path+"removal_list.npy")
    
    for num, i in enumerate(order_to_remove[:int(len(order_to_remove)*0.5)]):

        if os.path.isfile("./results/pruned/"+ model_name+"_"+str(num)+'.json'):
                
                    tmp = "./results/pruned/"+ model_name+"_"+str(num)+'.json'
                    
                    f = open(tmp)
                    res = json.load(f)
                    f.close()
                    
                    all_res.append(res["results"])
    
    all_res_model[model_name]=all_res
    
    
    tmp = "./results/unpruned/"+ model_name+'.json'
                    
    f = open(tmp)
    res = json.load(f)
    f.close()
    
    all_res_model[model_name+"_u"] = res["results"]


for dataset in ["boolq","rte","arc_challenge","arc_easy","openbookqa"]:

    for model_name in LMs:
        acc = [x[dataset]["acc,none"] for x in all_res_model[model_name]]
        plt.plot(range(1, len(acc)+1), acc, label=names[model_name], color=colors[model_name])
        
        plt.plot(range(1, len(acc)+1), [all_res_model[model_name+"_u"][dataset]["acc,none"]]*len(acc), color=colors[model_name],linestyle='--')
        
    
        
    plt.title(d_names[dataset], fontsize=20)
    plt.xlabel("# removed attention layers", fontsize=18)
    plt.ylabel("Accuracy", fontsize=18)
    plt.tick_params(axis='both', which='major', labelsize=16)
    plt.ylim(0,1)
    plt.savefig("./results/plots/effectiveness/"+dataset+".pdf", dpi=600, bbox_inches="tight")
    plt.show()
    plt.clf()

In [ ]:
for dataset in ["boolq","rte","arc_challenge","arc_easy","openbookqa"]:

    for model_name in LMs:
        acc = [x[dataset]["acc,none"] for x in all_res_model[model_name]]
        plt.plot(range(1, len(acc)+1), acc, label=names[model_name], color=colors[model_name])
        
        plt.plot(range(1, len(acc)+1), [all_res_model[model_name+"_u"][dataset]["acc,none"]]*len(acc), color=colors[model_name],linestyle='--')
        
    plt.title(d_names[dataset], fontsize=20)
    plt.xlabel("# removed attention layers", fontsize=18)
    plt.ylabel("Accuracy", fontsize=18)
    plt.xticks(np.arange(2, 17, 2))
    plt.tick_params(axis='both', which='major', labelsize=14)
    plt.ylim(0,1)
    plt.xlim(1,16)
    plt.savefig("./results/plots/effectiveness/"+dataset+".pdf", dpi=600, bbox_inches="tight")
    plt.show()
    plt.clf()